# Volume-Weighted Time Series Momentum Strategy

**Strategy Overview:**
- Entry: Positive momentum (30/60/90-day) AND volume > 20-day average
- Exit: Momentum reversal or trailing stop 2%
- Position sizing: Volatility-scaled (ATR-based)

**Reference:**
- Huang, Sangiorgi & Urquhart (SSRN 4825389)
- "Cryptocurrency Volume-Weighted Time Series Momentum"

In [ ]:
import sys

sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from medusa.backtest import VectorBTEngine
from medusa.strategies import VolumeMomentumStrategy

sns.set_style('darkgrid')
%matplotlib inline

## 1. Load Sample Data

For demonstration purposes, we'll generate synthetic price and volume data.
In production, replace this with actual data from CSV or kdb+.

In [ ]:
# Generate synthetic data for demo
np.random.seed(42)

n_days = 365 * 2  # 2 years
dates = pd.date_range('2024-01-01', periods=n_days, freq='D')

# Trending price with noise
trend = np.linspace(100, 150, n_days)
noise = np.random.randn(n_days) * 5
price = trend + noise

# Volume with spikes
base_volume = np.random.lognormal(10, 1, n_days)
volume_spikes = np.random.choice([1, 3], size=n_days, p=[0.9, 0.1])
volume = base_volume * volume_spikes

# Create OHLCV DataFrame
data = pd.DataFrame({
    'open': price * (1 + np.random.randn(n_days) * 0.01),
    'high': price * (1 + abs(np.random.randn(n_days)) * 0.015),
    'low': price * (1 - abs(np.random.randn(n_days)) * 0.015),
    'close': price,
    'volume': volume,
}, index=dates)

print(f"Data shape: {data.shape}")
print(f"Date range: {data.index.min()} to {data.index.max()}")
data.head()

## 2. Visualize Price and Volume

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price
axes[0].plot(data.index, data['close'], label='Close Price', color='blue')
axes[0].set_ylabel('Price')
axes[0].set_title('Price Chart')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(data.index, data['volume'], label='Volume', color='gray', alpha=0.5)
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')
axes[1].set_title('Volume Chart')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Initialize Strategy and Generate Signals

In [ ]:
# Initialize strategy
strategy = VolumeMomentumStrategy(
    lookback_windows=[30, 60, 90],
    volume_threshold=1.0,
    stop_loss_pct=0.02,
    vol_target=0.15,
    top_pct=0.3,
    bottom_pct=0.0,  # Long-only
)

print("Strategy parameters:")
print(f"  Lookback windows: {strategy.lookback_windows}")
print(f"  Volume threshold: {strategy.volume_threshold}")
print(f"  Stop loss: {strategy.stop_loss_pct:.1%}")
print(f"  Volatility target: {strategy.vol_target:.1%}")

In [ ]:
# Generate signals
entries, exits = strategy.generate_signals(data)

print(f"Total entry signals: {entries.sum()}")
print(f"Total exit signals: {exits.sum()}")
print(f"Signal frequency: {entries.sum() / len(data):.2%}")

## 4. Visualize Signals

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Price
ax.plot(data.index, data['close'], label='Close Price', color='blue', alpha=0.7)

# Entry signals
entry_prices = data.loc[entries, 'close']
ax.scatter(entry_prices.index, entry_prices, marker='^', color='green',
           label='Entry', s=100, zorder=5)

# Exit signals
exit_prices = data.loc[exits, 'close']
ax.scatter(exit_prices.index, exit_prices, marker='v', color='red',
           label='Exit', s=100, zorder=5)

ax.set_ylabel('Price')
ax.set_xlabel('Date')
ax.set_title('Volume Momentum Strategy Signals')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Run Backtest with VectorBT

In [ ]:
# Initialize backtest engine
engine = VectorBTEngine(
    initial_capital=100000.0,
    commission=0.001,  # 0.1% per trade
    slippage=0.0005,   # 0.05% slippage
    asset_class='crypto',
)

# Run backtest
portfolio = engine.run_signals(
    price=data['close'],
    entries=entries,
    exits=exits,
)

print("\nBacktest completed!")

## 6. Performance Metrics

In [ ]:
# Display stats
stats = portfolio.stats()
print("\n=== Performance Statistics ===")
print(stats)

# Key metrics
print("\n=== Key Metrics ===")
print(f"Total Return: {portfolio.total_return():.2%}")
print(f"Sharpe Ratio: {portfolio.sharpe_ratio():.2f}")
print(f"Max Drawdown: {portfolio.max_drawdown():.2%}")
print(f"Win Rate: {portfolio.win_rate():.2%}")
print(f"Number of Trades: {portfolio.trades.count()}")

## 7. Equity Curve

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Portfolio value
portfolio_value = portfolio.value()
ax.plot(portfolio_value.index, portfolio_value, label='Strategy', color='green', linewidth=2)

# Buy and hold baseline
buy_hold = (data['close'] / data['close'].iloc[0]) * engine.initial_capital
ax.plot(buy_hold.index, buy_hold, label='Buy & Hold', color='blue', alpha=0.5, linestyle='--')

ax.set_ylabel('Portfolio Value ($)')
ax.set_xlabel('Date')
ax.set_title('Equity Curve: Volume Momentum Strategy vs Buy & Hold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Trade Analysis

In [ ]:
# Get trades
trades = portfolio.trades.records_readable

if len(trades) > 0:
    print("\nFirst 10 trades:")
    print(trades.head(10))

    # Return distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Returns histogram
    axes[0].hist(trades['Return'], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    axes[0].axvline(trades['Return'].mean(), color='red', linestyle='--',
                    label=f'Mean: {trades["Return"].mean():.2%}')
    axes[0].set_xlabel('Return (%)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Trade Return Distribution')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Cumulative returns
    cumulative_returns = (1 + trades['Return']/100).cumprod()
    axes[1].plot(cumulative_returns.index, cumulative_returns, color='green', linewidth=2)
    axes[1].set_xlabel('Trade Number')
    axes[1].set_ylabel('Cumulative Return (x)')
    axes[1].set_title('Cumulative Returns per Trade')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No trades executed.")

## 9. Next Steps

1. **Load real data**: Replace synthetic data with actual crypto OHLCV from CSV or kdb+
2. **Parameter optimization**: Use `VectorBTEngine.optimize_params()` to find best lookback windows
3. **Walk-forward testing**: Use `VectorBTEngine.walk_forward()` for realistic out-of-sample testing
4. **Combine with filters**: Add VPIN filter to reduce entries during toxic flow periods
5. **Multi-asset**: Extend to trade multiple assets with rank-based position sizing